# Importare le librerie

In [1]:
import easyocr
reader = easyocr.Reader(["it"])
import os
import pandas as pd
import cv2
import numpy as np
import csv

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


# Definire le funzioni

In [2]:
def prima_lettura(png):
    result = reader.readtext(
        png,
        detail=1,
        paragraph=False,
        mag_ratio=2
    )
    return result

In [3]:
def zona(png):
    # Cercare le righe DESCRIZIONE e SUBTOTALE
    y_descrizione = None
    y_subtotale = None

    for bbox, txt, conf in result:
        testo = txt.upper()

        if "DESCRIZIONE" in testo:
            y_descrizione = min(p[1] for p in bbox)

        if "SUBTOTALE" in testo or "PLESSIVO" in testo: #Ultimamente legge male quella riga, ma forse ho capito che è perchè prima scaricavo l'immagine e la trasferivo sul PC, mentre dopo un pò ho provato a inviarla direttamente al PC
            y_subtotale = min(p[1] for p in bbox)

    # Tagliare solo la zona articoli
    img = cv2.imread(png)
    altezza, larghezza = img.shape[:2]

    zona_articoli = img[
        y_descrizione:y_subtotale,
        0:larghezza
    ]

    # Ingrandire
    def ingrandisci(img):
        return cv2.resize(
            img,
            None,
            fx=4,
            fy=4,
            interpolation=cv2.INTER_CUBIC
        )

    zona_articoli = ingrandisci(zona_articoli)
    return zona_articoli

In [4]:
def seconda_lettura(zona):
    ocr_ingrandito = reader.readtext(
        zona,
        detail=1,
        paragraph=False
        )
    return ocr_ingrandito

In [5]:
tolleranza_y = 50 #la lascio come variabile globale per utilizzarla in tutte le funzioni di raggruppamento per riga

In [6]:
def trasformazione_df(ocr_ingrandito):
    df_ocr_ingrandito = pd.DataFrame([
    {"Descrizione" : text,
     "Confidenza" : round(conf, 2),
     "x1" : round(bbox[0][0]), "y1" : round(bbox[0][1]),
     "x2" : round(bbox[1][0]), "y2" : round(bbox[1][1]),
     "x3" : round(bbox[2][0]), "y3" : round(bbox[2][1]),
     "x4" : round(bbox[3][0]), "y4" : round(bbox[3][1])}
     for bbox, text, conf in ocr_ingrandito]
    )

    df_ocr_ingrandito["Descrizione"] = df_ocr_ingrandito["Descrizione"].str.replace(" ", "")
    df_ocr_ingrandito_clean = df_ocr_ingrandito#.loc[righe_tenere]
    
    return df_ocr_ingrandito_clean

In [7]:
def prezzi(df_ocr_ingrandito_clean):
    df_prezzi = df_ocr_ingrandito_clean[3:].loc[df_ocr_ingrandito_clean["x1"] > 3000].sort_values("y1")
    df_prezzi["riga"] = (df_prezzi["y1"].diff().abs() >= tolleranza_y).cumsum()
    df_prezzi = df_prezzi.sort_values(["riga", "x1"])
    df_prezzi["Descrizione"] = (df_prezzi.groupby("riga")["Descrizione"].transform(" ".join))
    df_prezzi = df_prezzi.drop_duplicates("riga").drop(columns="riga")
    df_prezzi["Prezzo proposto"] = df_prezzi["Descrizione"].loc[df_prezzi["Descrizione"].str.contains(" ")]
    df_prezzi["Prezzo proposto"] = df_prezzi["Prezzo proposto"].str.replace({",": "",
                                                                            "=": "",
                                                                            "E": "",})
    df_prezzi["Prezzo proposto"] = df_prezzi["Prezzo proposto"].str.replace(" ", ".")
    df_prezzi.loc[df_prezzi["Descrizione"].str.contains(" "), "Descrizione"] = np.nan
    df_prezzi["Descrizione"] = df_prezzi["Descrizione"].str.replace(",", ".").astype(float)
    return df_prezzi

In [8]:
def iva(df_ocr_ingrandito_clean):
    df_iva = df_ocr_ingrandito_clean[3:].loc[(df_ocr_ingrandito_clean["x1"] > 2500) & (df_ocr_ingrandito_clean["x1"] <= 3000)].sort_values("y1")
    df_iva["Descrizione"] = df_iva["Descrizione"].str.replace({"22%" : "22",
                                                               "2%" : "2",
                                                            "229" : "22",
                                                            "10%" : "10",
                                                            "0%" : "10",
                                                            "109" : "10",
                                                            "5%" : "5",
                                                            "4%" : "4"}).astype(int)
    return df_iva

In [9]:
def articoli(df_ocr_ingrandito_clean):
    df_articoli = df_ocr_ingrandito_clean[3:].loc[df_ocr_ingrandito_clean["x1"] <= 2500].sort_values("y1")
    df_articoli["riga"] = (df_articoli["y1"].diff().abs() >= tolleranza_y).cumsum()
    df_articoli = df_articoli.sort_values(["riga", "x1"]) #senza questo le parole all'interno della riga erano in ordine sparso
    df_articoli["Descrizione"] = (df_articoli.groupby("riga")["Descrizione"].transform(" ".join))
    df_articoli = df_articoli.drop_duplicates("riga").drop(columns="riga")

    # tolgo le righe che non ci interessano
    df_articoli = df_articoli[~(df_articoli["Descrizione"].str.contains("Cad") & \
                            df_articoli["Descrizione"].str.contains("Pz|PZ"))]

    df_articoli = df_articoli[~df_articoli["Descrizione"].str.contains("kg|EUR/kg")].reset_index(drop=True)

    return df_articoli

In [10]:
def df_complessivo(df_articoli, df_iva, df_prezzi):
    df_finale = pd.concat([df_articoli["Descrizione"].reset_index(drop=True),
                        df_iva["Descrizione"].reset_index(drop=True),
                        df_prezzi["Descrizione"].reset_index(drop=True),
                        df_prezzi["Prezzo proposto"].reset_index(drop=True)],
                        axis = 1)
    df_finale.columns = ["Articolo", "IVA", "Prezzo", "Prezzo proposto"]
    df_finale["Articolo"] = df_finale["Articolo"].str.replace(",", ".") # questa l'ho aggiunta dopo, perchè ad esempio con Coca Cola 0,5 L poi nel CSV faceva casino perchè pensava che fosse una colonna in più
    return df_finale

In [11]:
def pulizia_df_complessivo(df_finale):
    df_finale["Sconto"] = pd.NA
    df_finale["Tipo sconto"] = pd.NA

    # poi posso sottrarre gli sconti e togliere le righe corrispondenti
    for i in range(len(df_finale)):
        articolo = str(df_finale.loc[i, "Articolo"])
        if "Offerta Lidl Plus" in articolo:
            tipo_sconto = "Offerta Lidl Plus"

        elif "Sconto Lidl Plus" in articolo:
            tipo_sconto = "Sconto Lidl Plus"

        elif "SCONTO" in articolo:
            tipo_sconto = "Prodotto in scadenza"

        else:
            tipo_sconto = pd.NA

        if pd.notna(tipo_sconto):
            sconto = df_finale.loc[i, "Prezzo"]
            df_finale.loc[i - 1, "Prezzo"] -= sconto
            df_finale.loc[i - 1, "Sconto"] = sconto
            df_finale.loc[i - 1, "Tipo sconto"] = tipo_sconto

    mask = ~df_finale["Articolo"].str.contains("Offerta Lidl Plus|Sconto Lidl Plus|SCONTO", na=False)
    df_finale = df_finale.loc[mask].reset_index(drop=True)

    # arrotondo perchè ho visto che nei calcoli a volte fa troppi decimali
    df_finale["Prezzo"] = df_finale["Prezzo"].round(2)

    # Poi devo togliere la riga di subtotale e degli sconti
    df_finale = df_finale[~df_finale["Articolo"].str.contains(r"VALOR[E F] SCONT[I T]|SUBTOTA[LE IF]")]
    return df_finale

In [12]:
def zona_info_scontrino(result):
    # Cercare le righe DESCRIZIONE e DOCUMENTO
    y_importo = None
    y_documento = None

    for bbox, txt, conf in result:
        testo = txt.upper()

        if "PAGATO" in testo: #"IMPORTO"
            y_importo = min(p[1] for p in bbox)

        if "DOCUMENTO" in testo:
            y_documento = min(p[1] for p in bbox)

    if y_importo is not None and y_documento is not None:

        risultati_filtrati = [
            (bbox, txt, conf)
            for bbox, txt, conf in result
            if y_importo <= min(p[1] for p in bbox) < y_documento
        ]
    return risultati_filtrati

In [13]:
def estrazione_data_ora(risultati_filtrati):
    df_risultati_filtrati = pd.DataFrame([
        {"Descrizione" : text,
        "Confidenza" : round(conf, 2),
        "x1" : round(bbox[0][0]), "y1" : round(bbox[0][1]),
        "x2" : round(bbox[1][0]), "y2" : round(bbox[1][1]),
        "x3" : round(bbox[2][0]), "y3" : round(bbox[2][1]),
        "x4" : round(bbox[3][0]), "y4" : round(bbox[3][1])}
        for bbox, text, conf in risultati_filtrati]
    )
    y_importo = df_risultati_filtrati["y1"][0]
    y_data_ora = y_importo + 100
    tolleranza_data_ora = 20

    df_risultati_filtrati = df_risultati_filtrati.loc[(df_risultati_filtrati["y1"] >= (y_data_ora - tolleranza_data_ora)) &
                            (df_risultati_filtrati["y1"] <= (y_data_ora + tolleranza_data_ora))].reset_index(drop=True)

    # La data viene spesso separata in più letture, ma a volte succede anche con l'ora. Quindi devo fare che la data è dal primo elemento fino all'anno
    indice_anno = df_risultati_filtrati.loc[df_risultati_filtrati["Descrizione"].str.contains("2026")].index[0] +1
    df_risultati_filtrati_data = df_risultati_filtrati.iloc[:indice_anno]
    df_risultati_filtrati_data = df_risultati_filtrati_data.sort_values("x1")
    Data = "-".join(df_risultati_filtrati_data["Descrizione"])

    df_risultati_filtrati_ora = df_risultati_filtrati.iloc[indice_anno:]
    df_risultati_filtrati_ora = df_risultati_filtrati_ora.sort_values("x1")
    Ora = ":".join(df_risultati_filtrati_ora["Descrizione"]).replace(" ", "")
    return Data, Ora

In [14]:
def aggiornamento_elenco(png, indice):
    with open("Elenco scontrini letti.csv", "a") as f:
        f.write(f"{png},")
        f.write(f"{indice},")
        f.write(f"{Data} {Ora}")
        f.write("\n")

# Processo principale

Processo:
1. Identificare gli scontrini di cui ho immagini
2. Controllare quali ho già letto nel file elenco scontrini
3. Processare gli scontrini nuovi
4. Scrivere nel file elenco scontrini che ho processato anche gli scontrini nuovi

In [17]:
cwd = os.getcwd()
raw_data = os.path.join(cwd, "raw_data")
png_files = [f for f in os.listdir(raw_data) if f.endswith(".png")]
processed_png = pd.read_csv("Elenco scontrini letti.csv")["Scontrino"].tolist()
png_to_process = [f for f in png_files if f not in processed_png]

if len(png_to_process) == 0:
    print("Non ci sono file da processare")
else:
    print("Devo processare i file", png_to_process)

ultimo_indice = pd.read_csv("Elenco scontrini letti.csv")["Indice"].tolist()[-1]
indici_to_process = list(range(ultimo_indice +1, ultimo_indice +1 +len(png_to_process)))

Devo processare i file ['2026.06.06_18002073320260606418073.png', '2026.06.06_18002073320260606418077.png', '2026.06.13_1800411812026061342409.png', '2026.06.14_18002073320260614420505.png', '2026.06.28_18002073120260628320730.png', '2026.06.28_18002073120260628320731.png', '2026.07.03_18002073120260703323296.png', '2026.07.03_18002073120260703323297.png', '2026.07.11_18002073120260711326754.png', '2026.07.11_18002073120260711326755.png', '2026.07.18_18002073320260718433711.png', '2026.07.18_18002073320260718433712.png', '2026.07.26_18002073420260726342668.png', '2026.08.02_18002073320260802440671.png', '2026.08.08_18002073320260808443142.png', '2026.08.14_18002073320260814445794.jpg.png', '2026.08.21_18002073320260821448678.jpg.png', '2026.08.28_18002073120260828348741.png']


Qui eseguo tutto il codice, con alla fine l'aggiunta della colonna indice nel ciclo for

In [18]:
for png,indice in zip(png_to_process, indici_to_process):
    print("Inizio a processare il file", png)
    png_da_leggere = os.path.join(cwd, "raw_data", png)
    result = prima_lettura(png_da_leggere)
    zona_articoli = zona(png_da_leggere)
    ocr_ingrandito = seconda_lettura(zona_articoli)
    df_ocr_ingrandito_clean = trasformazione_df(ocr_ingrandito)
    df_prezzi = prezzi(df_ocr_ingrandito_clean)
    df_iva = iva(df_ocr_ingrandito_clean)
    df_articoli = articoli(df_ocr_ingrandito_clean)
    df_finale = df_complessivo(df_articoli, df_iva, df_prezzi)
    df_finale = pulizia_df_complessivo(df_finale)
    df_finale["Indice scontrino"] = indice
    df_finale.to_csv("Database.csv", mode="a", header=False, index=False)
    risultati_filtrati = zona_info_scontrino(result)
    Data, Ora = estrazione_data_ora(risultati_filtrati)
    aggiornamento_elenco(png, indice)
    print("Completato", indici_to_process.index(indice) +1, "di", len(indici_to_process))

Inizio a processare il file 2026.06.06_18002073320260606418073.png


c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completato 1 di 18
Inizio a processare il file 2026.06.06_18002073320260606418077.png


c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completato 2 di 18
Inizio a processare il file 2026.06.13_1800411812026061342409.png


...

Completato 16 di 18
Inizio a processare il file 2026.08.21_18002073320260821448678.jpg.png


c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completato 17 di 18
Inizio a processare il file 2026.08.28_18002073120260828348741.png


c:\Users\Computer\miniconda3\envs\OCR\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completato 18 di 18


# Correzioni manuali

## Creazione

In [67]:
# database = pd.read_csv("Database.csv")
# database.loc[database["Prezzo"].isna()].to_csv("Corretti manualmente.csv", index=False)

## Aggiornamento

In [16]:
database = pd.read_csv("Database.csv")
database = database.loc[database["Prezzo"].isna()]

# Aggiungo un nuovo articolo fittizio
# database.loc[len(database)] = ["Nuovo articolo", 4, 0.50, np.nan, 10]

In [17]:
articoli_corretti = pd.read_csv("Corretti manualmente.csv")
articoli_da_aggiungere = database.merge(articoli_corretti, how="outer", indicator=True)
articoli_da_aggiungere = articoli_da_aggiungere[articoli_da_aggiungere["_merge"] == "left_only"].drop(columns="_merge")

ParserError: Error tokenizing data. C error: Expected 5 fields in line 11, saw 6


In [ ]:
with open("Corretti manualmente.csv", "a", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    for riga in articoli_da_aggiungere.itertuples(index=False, name=None):
        writer.writerow(riga)

# Debug

In [ ]:
# y_descrizione = None
# y_subtotale = None

# for bbox, txt, conf in result:
#     testo = txt.upper()

#     if "DESCRIZIONE" in testo:
#         y_descrizione = min(p[1] for p in bbox)

#     if "SUBTOTALE" in testo or "PLESSIVO" in testo: #Ho aggiunto una lettura alternativa
#         y_subtotale = min(p[1] for p in bbox)

In [ ]:
# print(y_descrizione, y_subtotale)

1054 2888


In [ ]:
# df_result = pd.DataFrame([
#     {"Descrizione" : text,
#      "Confidenza" : round(conf, 2),
#      "x1" : round(bbox[0][0]), "y1" : round(bbox[0][1]),
#      "x2" : round(bbox[1][0]), "y2" : round(bbox[1][1]),
#      "x3" : round(bbox[2][0]), "y3" : round(bbox[2][1]),
#      "x4" : round(bbox[3][0]), "y4" : round(bbox[3][1])}
#      for bbox, text, conf in result]
#     )

In [ ]:
# df_result.loc[df_result["Descrizione"] == "DESCRIZIONE"]

,Descrizione,Confidenza,x1,y1,x2,y2,x3,y3,x4,y4
15,DESCRIZIONE,1.0,55,1054,333,1054,333,1099,55,1099


In [ ]:
# df_result.loc[df_result["Descrizione"].str.contains("OMPLESSIVO")]

,Descrizione,Confidenza,x1,y1,x2,y2,x3,y3,x4,y4


In [ ]:
# df_result.iloc[120:131]

,Descrizione,Confidenza,x1,y1,x2,y2,x3,y3,x4,y4
120,CAFFE,1.00,399,2760,528,2760,528,2804,399,2804
121,09,0.88,690,2760,748,2760,748,2804,690,2804
122,0,0.58,911,2760,949,2760,949,2804,911,2804
123,70,1.00,963,2760,1021,2760,1021,2804,963,2804
124,Cad,1.00,55,2811,139,2811,139,2856,55,2856
125,35,1.00,197,2804,256,2804,256,2857,197,2857
126,VALORE SCONTI,1.00,54,2849,380,2849,380,2903,54,2903
127,00,1.00,885,2857,943,2857,943,2902,885,2902
128,58,1.00,892,2902,943,2902,943,2947,892,2947
129,SUfKCTACONPLESSIVO,0.03,40,2888,518,2888,518,3020,40,3020


In [ ]:
# df_finale

,Articolo,IVA,Prezzo,Prezzo proposto,Sconto,Tipo sconto,Indice scontrino
0,LATTE PASTORIZZATO,4.0,1.25,NaN,<NA>,<NA>,19
1,MELONE GIALLO,4.0,1.00,NaN,<NA>,<NA>,19
2,PESCHE 1KG,4.0,1.79,NaN,<NA>,<NA>,19
3,MOZZAREL NO MANGIMI,4.0,1.49,NaN,<NA>,<NA>,19
4,INSALATA SFIZIOSA,4.0,0.49,NaN,0.5,Prodotto in scadenza,19
5,POMODORO COKTAIL,4.0,1.99,NaN,<NA>,<NA>,19
6,CROISSANT NOCCIOLA,10.0,0.59,NaN,<NA>,<NA>,19
7,SENZA PECCATO MOUSSE,10.0,1.27,NaN,0.32,Prodotto in scadenza,19
8,YOGURT INTERO LIMONE,10.0,0.70,NaN,<NA>,<NA>,19
9,YOGURT INt FRAGOLA,10.0,0.70,NaN,<NA>,<NA>,19
